# DGCA Fusion на RESD: rubert-tiny2 (text) + WavLM (audio)

Pipeline:
1. Загружаем `Aniemore/resd` с HuggingFace (колонки: `speech`, `emotion`).
2. Транскрибируем аудио через Whisper (`artyomboyko/whisper-small-ru-v2`) один раз перед обучением.
3. BERT backbone: `Aniemore/rubert-tiny2-russian-emotion-detection` → CLS embedding.
4. WavLM backbone: `Aniemore/wavlm-emotion-russian-resd` → mean-pooled embedding.
5. DGCA Fusion → 7-классовая классификация (RESD labels).

```
h_text  = BERT(X)[CLS]          ∈ R^D_text
h_audio = MeanPool(WavLM(wav))  ∈ R^D_audio
```

## 1. Install & clone

In [ ]:
import subprocess, sys, os

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'datasets', 'peft>=0.10', 'soundfile', 'torchaudio',
    'librosa', 'scikit-learn', 'tqdm', 'pyyaml',
], check=True)

REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Done. CWD:', os.getcwd())

## 2. Imports & config

In [ ]:
import warnings, pathlib, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoFeatureExtractor,
    pipeline, get_linear_schedule_with_warmup,
)
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, classification_report, accuracy_score
from tqdm.auto import tqdm
import soundfile as sf
import librosa
warnings.filterwarnings('ignore')

# ── модели ────────────────────────────────────────────────────────────────────
WHISPER_MODEL = 'artyomboyko/whisper-small-ru-v2'
BERT_MODEL    = 'Aniemore/rubert-tiny2-russian-emotion-detection'
WAVLM_MODEL   = 'Aniemore/wavlm-emotion-russian-resd'

# ── пути ──────────────────────────────────────────────────────────────────────
OUT_DIR = pathlib.Path('/kaggle/working')

# ── гиперпараметры ────────────────────────────────────────────────────────────
FUSION_DIM   = 512
NUM_HEADS    = 4
BATCH_SIZE   = 8
LR_FUSION    = 1e-4
LR_BACKBONE  = 1e-5
WEIGHT_DECAY = 1e-2
WARMUP_STEPS = 50
MAX_STEPS    = 2000
EVAL_EVERY   = 50
ES_PATIENCE  = 8
MAX_TEXT_LEN = 128
MAX_AUDIO_S  = 10.0
SR_TARGET    = 16_000
SEED         = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

RESD_LABEL2ID = {
    'happiness': 0, 'sadness': 1, 'anger': 2,
    'fear': 3, 'disgust': 4, 'enthusiasm': 5, 'neutral': 6,
}
RESD_LABELS = ['happiness', 'sadness', 'anger', 'fear', 'disgust', 'enthusiasm', 'neutral']
NUM_CLASSES = len(RESD_LABELS)

## 3. Загрузка данных RESD

In [ ]:
print('Loading Aniemore/resd from HuggingFace...')
resd = load_dataset('Aniemore/resd')
print(resd)

def hf_split_to_records(split):
    records = []
    for item in split:
        label_str = item['emotion']
        if label_str not in RESD_LABEL2ID:
            continue
        records.append({
            'audio':  item['speech']['array'],
            'sr':     item['speech']['sampling_rate'],
            'label':  RESD_LABEL2ID[label_str],
        })
    return records

all_train_recs = hf_split_to_records(resd['train'])
test_recs      = hf_split_to_records(resd['test'])

train_recs, val_recs = train_test_split(
    all_train_recs, test_size=0.20, random_state=SEED,
    stratify=[r['label'] for r in all_train_recs],
)
print(f'Train: {len(train_recs)}  Val: {len(val_recs)}  Test: {len(test_recs)}')

## 4. Whisper smoke test (5 сэмплов)

In [ ]:
import IPython.display as ipd

print(f'Loading Whisper: {WHISPER_MODEL}')
asr = pipeline(
    'automatic-speech-recognition', model=WHISPER_MODEL,
    device=0 if torch.cuda.is_available() else -1,
    generate_kwargs={'language': 'russian', 'task': 'transcribe'},
)

print('\n=== Whisper smoke test (5 samples) ===')
for i, rec in enumerate(train_recs[:5]):
    wav = rec['audio'].astype('float32')
    sr  = rec['sr']
    if sr != SR_TARGET:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
    result = asr(wav)   # передаём numpy array напрямую
    label  = RESD_LABELS[rec['label']]
    print(f'[{i}] {label:12s} | {result["text"]}')
    display(ipd.Audio(wav, rate=SR_TARGET))
print()

## 5. Транскрипция всего датасета через Whisper

In [ ]:
def transcribe_records(records, desc='Transcribing'):
    out = []
    for rec in tqdm(records, desc=desc):
        wav = rec['audio'].astype('float32')
        sr  = rec['sr']
        if sr != SR_TARGET:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR_TARGET)
        result = asr(wav)   # numpy array напрямую
        out.append({
            'text':  result['text'],
            'audio': wav,
            'label': rec['label'],
        })
    return out

train_recs = transcribe_records(train_recs, 'Train ASR')
val_recs   = transcribe_records(val_recs,   'Val ASR')
test_recs  = transcribe_records(test_recs,  'Test ASR')
print('Transcription done.')
print(f'Example: "{train_recs[0]["text"]}"  → {RESD_LABELS[train_recs[0]["label"]]}')

## 6. Загрузка backbone моделей

In [ ]:
print(f'Loading BERT: {BERT_MODEL}')
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_backbone  = AutoModel.from_pretrained(BERT_MODEL).to(device)
BERT_DIM = bert_backbone.config.hidden_size
print(f'  BERT hidden_size: {BERT_DIM}')

print(f'Loading WavLM: {WAVLM_MODEL}')
wavlm_processor = AutoFeatureExtractor.from_pretrained(WAVLM_MODEL)
wavlm_backbone  = AutoModel.from_pretrained(WAVLM_MODEL).to(device)
WAVLM_DIM = wavlm_backbone.config.hidden_size
print(f'  WavLM hidden_size: {WAVLM_DIM}')

## 7. DGCA Fusion модуль

In [ ]:
class DGCAFusion(nn.Module):
    """
    Dimension-Wise Gated Cross-Attention fusion (DGCA, WWW'25).
    Адаптация для Text (BERT) + Audio (WavLM).

    Шаги:
      1. Projection:   T = W_t·h_text + b_t,  A = W_a·h_audio + b_a   → R^D
      2. Cross-Attn:   T_ref = LN(T + MHA(T,A,A)), A_ref = LN(A + MHA(A,T,T))
      3. Gating:       α_t(d) = softmax([G_t(d), G_a(d)])[0] per dimension d
      4. Fusion:       F(d) = α_t(d)·T_ref(d) + α_a(d)·A_ref(d)
      5. Classify:     logits = W_c·F + b_c
    """
    def __init__(self, d_text, d_audio, D=512, num_heads=4, num_classes=5, dropout=0.1):
        super().__init__()
        # 1. Projection
        self.proj_text  = nn.Linear(d_text,  D)
        self.proj_audio = nn.Linear(d_audio, D)

        # 2. Cross-Attention (bidirectional)
        self.mha_t2a = nn.MultiheadAttention(D, num_heads, batch_first=True, dropout=dropout)
        self.mha_a2t = nn.MultiheadAttention(D, num_heads, batch_first=True, dropout=dropout)
        self.ln_text  = nn.LayerNorm(D)
        self.ln_audio = nn.LayerNorm(D)

        # 3. Dimension-wise gating
        self.gate_text  = nn.Linear(D, D)
        self.gate_audio = nn.Linear(D, D)

        # 5. Classifier
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(D, num_classes)

    def forward(self, h_text, h_audio):
        """
        h_text  : (B, d_text)
        h_audio : (B, d_audio)
        returns : logits (B, num_classes)
        """
        # 1. Projection → (B, 1, D)
        T = self.proj_text(h_text).unsqueeze(1)
        A = self.proj_audio(h_audio).unsqueeze(1)

        # 2. Bidirectional cross-attention
        T_ca, _ = self.mha_t2a(T, A, A)          # text queries audio
        T_ref   = self.ln_text(T + T_ca)          # residual + LN
        A_ca, _ = self.mha_a2t(A, T, T)          # audio queries text
        A_ref   = self.ln_audio(A + A_ca)

        T_ref = T_ref.squeeze(1)                  # (B, D)
        A_ref = A_ref.squeeze(1)

        # 3. Dimension-wise gating logits
        G_t = self.gate_text(T_ref)               # (B, D)
        G_a = self.gate_audio(A_ref)              # (B, D)

        # softmax across modalities per dimension
        gates   = F.softmax(torch.stack([G_t, G_a], dim=-1), dim=-1)  # (B, D, 2)
        alpha_t = gates[..., 0]                   # (B, D)
        alpha_a = gates[..., 1]

        # 4. Dimension-wise fusion
        fused = alpha_t * T_ref + alpha_a * A_ref  # (B, D)

        # 5. Classification
        return self.classifier(self.dropout(fused))


fusion = DGCAFusion(
    d_text=BERT_DIM, d_audio=WAVLM_DIM,
    D=FUSION_DIM, num_heads=NUM_HEADS, num_classes=NUM_CLASSES,
).to(device)

total = sum(p.numel() for p in fusion.parameters())
print(f'DGCA fusion params: {total:,}')

## 8. Dataset & DataLoader

In [ ]:
MAX_AUDIO_LEN = int(MAX_AUDIO_S * SR_TARGET)


class MultimodalDataset(Dataset):
    def __init__(self, records, tokenizer, max_text_len=MAX_TEXT_LEN):
        self.records   = records
        self.tokenizer = tokenizer
        self.max_text_len = max_text_len

    def __len__(self): return len(self.records)

    def __getitem__(self, i):
        r = self.records[i]
        enc = self.tokenizer(
            r['text'], truncation=True, padding='max_length',
            max_length=self.max_text_len, return_tensors='pt'
        )
        wav = r['audio'].astype('float32')
        if len(wav) > MAX_AUDIO_LEN:
            wav = wav[:MAX_AUDIO_LEN]
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'audio':          torch.tensor(wav, dtype=torch.float32),
            'label':          torch.tensor(r['label'], dtype=torch.long),
        }


def collate_fn(batch):
    """Pad audio sequences to max length in batch."""
    max_len = max(b['audio'].shape[0] for b in batch)
    audios  = torch.zeros(len(batch), max_len)
    masks   = torch.zeros(len(batch), max_len)
    for i, b in enumerate(batch):
        L = b['audio'].shape[0]
        audios[i, :L] = b['audio']
        masks[i,  :L] = 1.0
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'audio':          audios,
        'audio_mask':     masks,
        'label':          torch.stack([b['label']          for b in batch]),
    }


train_ds = MultimodalDataset(train_recs, bert_tokenizer)
val_ds   = MultimodalDataset(val_recs,   bert_tokenizer)
test_ds  = MultimodalDataset(test_recs,  bert_tokenizer)

train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_ld   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True, collate_fn=collate_fn)
test_ld  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True, collate_fn=collate_fn)

print(f'Train batches: {len(train_ld)}  Val: {len(val_ld)}  Test: {len(test_ld)}')

## 9. Encoder helpers

In [ ]:
def encode_text(input_ids, attention_mask):
    """BERT → CLS embedding: (B, BERT_DIM)"""
    out = bert_backbone(input_ids=input_ids, attention_mask=attention_mask)
    return out.last_hidden_state[:, 0, :]   # CLS token


def encode_audio(audio, audio_mask):
    """
    WavLM → mean-pooled embedding: (B, WAVLM_DIM).
    Обрабатываем по одному сэмплу через процессор, затем батчим.
    """
    embeddings = []
    for i in range(audio.shape[0]):
        wav_np = audio[i][audio_mask[i].bool()].cpu().numpy()
        inputs = wavlm_processor(wav_np, sampling_rate=SR_TARGET, return_tensors='pt')
        input_values = inputs['input_values'].to(device)
        hidden = wavlm_backbone(input_values).last_hidden_state  # (1, T, D)
        embeddings.append(hidden.mean(dim=1).squeeze(0))         # (D,)
    return torch.stack(embeddings)                               # (B, D)

## 10. Обучение

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': bert_backbone.parameters(),  'lr': LR_BACKBONE},
    {'params': wavlm_backbone.parameters(), 'lr': LR_BACKBONE},
    {'params': fusion.parameters(),         'lr': LR_FUSION},
], weight_decay=WEIGHT_DECAY)

scheduler_warmup = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=MAX_STEPS)
scheduler_plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2)

criterion = nn.CrossEntropyLoss()

best_val_wacc = -1.0
es_counter    = 0
global_step   = 0
history       = []
train_iter    = iter(train_ld)


@torch.no_grad()
def evaluate(loader):
    bert_backbone.eval(); wavlm_backbone.eval(); fusion.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in loader:
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        audio = batch['audio'].to(device)
        amask = batch['audio_mask'].to(device)
        labs  = batch['label'].to(device)
        h_text  = encode_text(ids, mask)
        h_audio = encode_audio(audio, amask)
        logits  = fusion(h_text, h_audio)
        total_loss  += criterion(logits, labs).item() * len(labs)
        preds_all.append(logits.argmax(-1).cpu().numpy())
        labels_all.append(labs.cpu().numpy())
    val_loss = total_loss / len(loader.dataset)
    val_wacc = balanced_accuracy_score(
        np.concatenate(labels_all), np.concatenate(preds_all))
    bert_backbone.train(); wavlm_backbone.train(); fusion.train()
    return val_loss, val_wacc


print(f'Training  max_steps={MAX_STEPS}  eval_every={EVAL_EVERY}  es={ES_PATIENCE}  save_by=val_wacc')
print(f'LR backbone={LR_BACKBONE}  LR fusion={LR_FUSION}  batch={BATCH_SIZE}')
print('-' * 70)

bert_backbone.train(); wavlm_backbone.train(); fusion.train()
running_loss = 0.0

while global_step < MAX_STEPS:
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_ld)
        batch = next(train_iter)

    ids   = batch['input_ids'].to(device)
    mask  = batch['attention_mask'].to(device)
    audio = batch['audio'].to(device)
    amask = batch['audio_mask'].to(device)
    labs  = batch['label'].to(device)

    optimizer.zero_grad()
    h_text  = encode_text(ids, mask)
    h_audio = encode_audio(audio, amask)
    logits  = fusion(h_text, h_audio)
    loss    = criterion(logits, labs)
    loss.backward()
    nn.utils.clip_grad_norm_(
        list(bert_backbone.parameters()) +
        list(wavlm_backbone.parameters()) +
        list(fusion.parameters()), 1.0)
    optimizer.step()
    scheduler_warmup.step()
    running_loss += loss.item()
    global_step  += 1

    if global_step % EVAL_EVERY == 0:
        avg_loss = running_loss / EVAL_EVERY
        val_loss, val_wacc = evaluate(val_ld)
        prev_lr  = optimizer.param_groups[-1]['lr']
        scheduler_plateau.step(val_wacc)
        cur_lr   = optimizer.param_groups[-1]['lr']
        running_loss = 0.0

        is_best = val_wacc > best_val_wacc
        if is_best:
            best_val_wacc = val_wacc
            es_counter    = 0
            torch.save({
                'fusion':         fusion.state_dict(),
                'bert_backbone':  bert_backbone.state_dict(),
                'wavlm_backbone': wavlm_backbone.state_dict(),
                'step':           global_step,
                'val_wacc':       val_wacc,
            }, str(OUT_DIR / 'best_dgca_resd.pt'))
        else:
            es_counter += 1

        lr_info = f'{cur_lr:.2e}' + (' ↓' if cur_lr < prev_lr else '')
        history.append((global_step, avg_loss, val_loss, val_wacc))
        print(
            f'Step {global_step:6d}  train={avg_loss:.4f}  '
            f'val_loss={val_loss:.4f}  val_wacc={val_wacc:.4f}  '
            f'lr={lr_info}' + ('  *' if is_best else ''),
            flush=True,
        )

        if es_counter >= ES_PATIENCE:
            print(f'Early stopping at step {global_step}')
            break

print(f'\nBest val_wacc: {best_val_wacc:.4f}')
print(f'Saved → {OUT_DIR / "best_dgca_resd.pt"}')

## 11. Кривые обучения

In [ ]:
import matplotlib.pyplot as plt

steps      = [h[0] for h in history]
tr_losses  = [h[1] for h in history]
val_losses = [h[2] for h in history]
val_waccs  = [h[3] for h in history]
best_step  = steps[int(np.argmax(val_waccs))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(steps, tr_losses,  label='train loss', color='steelblue')
ax1.plot(steps, val_losses, label='val loss',   color='darkorange')
ax1.axvline(best_step, color='red', linestyle='--', alpha=0.6, label=f'best={best_step}')
ax1.set_xlabel('Step'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend()

ax2.plot(steps, val_waccs, color='green', label='val wacc')
ax2.axvline(best_step, color='red', linestyle='--', alpha=0.6)
ax2.set_xlabel('Step'); ax2.set_ylabel('Weighted Accuracy')
ax2.set_title('Val Weighted Accuracy'); ax2.legend()

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'dgca_resd_curves.png'), dpi=150)
plt.show()

## 12. Финальная оценка на test

In [ ]:
ckpt_best = torch.load(str(OUT_DIR / 'best_dgca_resd.pt'), map_location=device)
fusion.load_state_dict(ckpt_best['fusion'])
bert_backbone.load_state_dict(ckpt_best['bert_backbone'])
wavlm_backbone.load_state_dict(ckpt_best['wavlm_backbone'])
print(f'Loaded best checkpoint from step {ckpt_best["step"]}')

bert_backbone.eval(); wavlm_backbone.eval(); fusion.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for batch in tqdm(test_ld, desc='Test'):
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        audio = batch['audio'].to(device)
        amask = batch['audio_mask'].to(device)
        h_text  = encode_text(ids, mask)
        h_audio = encode_audio(audio, amask)
        preds_all.append(fusion(h_text, h_audio).argmax(-1).cpu().numpy())
        labels_all.append(batch['label'].numpy())

preds  = np.concatenate(preds_all)
labels = np.concatenate(labels_all)

print('\n=== Test Results ===')
print(f'Accuracy          : {accuracy_score(labels, preds):.4f}')
print(f'Weighted Accuracy  : {balanced_accuracy_score(labels, preds):.4f}')
print()
print(classification_report(labels, preds, target_names=RESD_LABELS, zero_division=0))